# ⚡ POWERGUARD AI: Electricity Consumption Anomaly & Theft Detection
### Final-Year Engineering & Hackathon 2026 Machine Learning Pipeline

**Objective**: Build a production-grade, reproducible unsupervised anomaly detection pipeline for smart meter electrical telemetry (Voltage, Current, Active Power, Power Factor, Frequency).

**Dataset**: UCI Individual Household Electric Power Consumption (Available on Kaggle: `uciml/electric-power-consumption`)

**Key Scientific Principles**:
1. **No Fabricated Labels**: Raw electricity meter readings represent genuine household consumption without verified theft ground truth. The model is formulated as an **unsupervised Isolation Forest** to isolate anomalous physical parameters rather than pretending artificial labels are verified theft.
2. **Leakage Prevention**: We enforce a **strict chronological train/test split** (80/20) to prevent time-series lookahead bias.
3. **Physical & Cyclic Features**: Feature engineering accounts for grid AC physics ($S = V \times I$, $\text{PF} = P/S$) and 24-hour diurnal human usage cycles.

## Step 1: Environment Setup & Library Imports

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import joblib

print("Libraries successfully imported!")

## Step 2: Load & Inspect the Kaggle Dataset
On Kaggle Notebooks, add the dataset `uciml/electric-power-consumption`. If running offline or in standalone mode, the notebook provides calibrated physical simulation mirroring the UCI distribution.

In [ ]:
kaggle_path = "/kaggle/input/electric-power-consumption/household_power_consumption.txt"
local_path = "household_power_consumption.txt"

if os.path.exists(kaggle_path):
    print("Loading dataset from Kaggle input path...")
    df = pd.read_csv(kaggle_path, sep=';', na_values=['?'], low_memory=False)
    df['datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], format='%d/%m/%Y %H:%M:%S', errors='coerce')
    df = df.dropna(subset=['datetime']).sort_values('datetime').reset_index(drop=True)
    df['voltage'] = pd.to_numeric(df['Voltage'], errors='coerce')
    df['current'] = pd.to_numeric(df['Global_intensity'], errors='coerce')
    df['power'] = pd.to_numeric(df['Global_active_power'], errors='coerce') * 1000.0 # to Watts
    df['frequency'] = 50.0 + np.random.normal(0, 0.05, len(df))
    df['power_factor'] = (df['power'] / (df['voltage'] * df['current'] + 1e-5)).clip(0.5, 0.99)
    df['is_anomaly_benchmark'] = 0
    df = df[['datetime', 'voltage', 'current', 'power', 'power_factor', 'frequency', 'is_anomaly_benchmark']]
else:
    print("Kaggle path not detected. Generating calibrated physical dataset matching UCI statistics...")
    np.random.seed(42)
    n_samples = 20000
    start_time = datetime(2025, 1, 1, 0, 0, 0)
    timestamps = [start_time + timedelta(minutes=15 * i) for i in range(n_samples)]
    hours = np.array([t.hour + t.minute / 60.0 for t in timestamps])
    
    morning_peak = np.exp(-((hours - 8.0) ** 2) / 6.0) * 8.0
    evening_peak = np.exp(-((hours - 20.0) ** 2) / 8.0) * 12.0
    current = np.clip(2.0 + morning_peak + evening_peak + np.random.normal(0, 0.8, n_samples), 0.2, 45.0)
    voltage = np.clip(230.0 - (current * 0.12) + np.random.normal(0, 1.8, n_samples), 205.0, 255.0)
    power_factor = np.clip(0.92 + np.random.normal(0, 0.03, n_samples), 0.70, 0.99)
    power = voltage * current * power_factor
    frequency = np.clip(50.0 + np.random.normal(0, 0.04, n_samples), 49.7, 50.3)
    
    # Benchmark fault injections (3% of samples)
    ground_truth = np.zeros(n_samples, dtype=int)
    anomaly_indices = np.random.choice(n_samples, size=int(n_samples * 0.03), replace=False)
    for idx in anomaly_indices:
        current[idx] = np.random.uniform(0.01, 0.2) # Shunt bypass / tap
        power_factor[idx] = np.random.uniform(0.3, 0.55)
        power[idx] = voltage[idx] * current[idx] * power_factor[idx]
        ground_truth[idx] = 1
        
    df = pd.DataFrame({
        'datetime': timestamps,
        'voltage': voltage,
        'current': current,
        'power': power,
        'power_factor': power_factor,
        'frequency': frequency,
        'is_anomaly_benchmark': ground_truth
    })

print(f"Dataset shape: {df.shape}")
print(df.head())

## Step 3: Data Cleaning & Preprocessing
Handle missing values, enforce physical boundary constraints, and sort chronologically.

In [ ]:
# Forward-fill and backward-fill short sensor dropouts
df = df.ffill().bfill()

# Physical bounds clipping
df['voltage'] = df['voltage'].clip(150.0, 300.0)
df['current'] = df['current'].clip(0.0, 100.0)
df['power'] = df['power'].clip(0.0, 35000.0)
df['power_factor'] = df['power_factor'].clip(0.05, 1.0)
df['frequency'] = df['frequency'].clip(45.0, 65.0)

print("Data cleaning complete. Summary statistics:")
print(df.describe().T[['mean', 'std', 'min', 'max']])

## Step 4: Feature Engineering
Create apparent power ($S = V \times I / 1000$) in kVA and cyclic representations of the hour of the day using sine/cosine transforms to avoid integer boundary discontinuities.

In [ ]:
FEATURE_COLUMNS = [
    'voltage',
    'current',
    'power',
    'power_factor',
    'frequency',
    'apparent_power',
    'hour_sin',
    'hour_cos'
]

df['apparent_power'] = (df['voltage'] * df['current']) / 1000.0
hours = df['datetime'].dt.hour + (df['datetime'].dt.minute / 60.0)
df['hour_sin'] = np.sin(2 * np.pi * hours / 24.0)
df['hour_cos'] = np.cos(2 * np.pi * hours / 24.0)

print("Engineered features:", FEATURE_COLUMNS)

## Step 5: Chronological Train/Test Split (Preventing Data Leakage)
In time-series smart meter streams, random shuffling leaks temporal auto-correlation. We partition the earliest 80% as training data and reserve the final 20% strictly for out-of-sample evaluation.

In [ ]:
split_idx = int(len(df) * 0.8)
train_df = df.iloc[:split_idx].copy()
test_df = df.iloc[split_idx:].copy()

scaler = StandardScaler()
X_train = scaler.fit_transform(train_df[FEATURE_COLUMNS])
X_test = scaler.transform(test_df[FEATURE_COLUMNS])
y_test = test_df['is_anomaly_benchmark'].values

print(f"Train samples: {X_train.shape[0]} | Test samples: {X_test.shape[0]}")

## Step 6: Train Isolation Forest Model
Isolation Forest builds an ensemble of isolation trees that isolate rare, multi-parameter anomalies near the root of the tree.

In [ ]:
iso_forest = IsolationForest(
    n_estimators=150,
    contamination=0.03,
    random_state=42,
    n_jobs=-1
)

iso_forest.fit(X_train)
test_scores = iso_forest.decision_function(X_test)
test_preds = (iso_forest.predict(X_test) == -1).astype(int)

print("Isolation Forest successfully trained!")
print(f"Test anomalies detected: {test_preds.sum()} out of {len(test_preds)} samples ({test_preds.mean()*100:.2f}%)")

## Step 7: Baseline Comparison & Evaluation
We evaluate Isolation Forest against a standard single-parameter statistical Z-Score threshold detector ($|Z| > 2.8$).

In [ ]:
# Baseline Z-Score detector
baseline_z = np.max(np.abs(X_test[:, :4]), axis=1)
baseline_preds = (baseline_z > 2.8).astype(int)

if y_test.sum() > 0:
    iso_auc = roc_auc_score(y_test, -test_scores)
    baseline_auc = roc_auc_score(y_test, baseline_z)
    iso_f1 = f1_score(y_test, test_preds, zero_division=0)
    baseline_f1 = f1_score(y_test, baseline_preds, zero_division=0)
    
    print("=" * 40)
    print(f"Isolation Forest ROC-AUC : {iso_auc:.4f}")
    print(f"Baseline Z-Score ROC-AUC : {baseline_auc:.4f}")
    print(f"Isolation Forest F1-Score: {iso_f1:.4f}")
    print(f"Baseline Z-Score F1-Score: {baseline_f1:.4f}")
    print("=" * 40)
    print("\nClassification Report (Isolation Forest):")
    print(classification_report(y_test, test_preds, target_names=['Normal', 'Anomaly']))
else:
    print("Unsupervised evaluation: anomaly score percentiles:")
    print(pd.Series(test_scores).describe())

## Step 8: Visualizing Detected Anomalies

In [ ]:
plt.figure(figsize=(14, 5))
subset_len = min(500, len(test_df))
times = test_df['datetime'].iloc[:subset_len]
currents = test_df['current'].iloc[:subset_len]
anomalies = test_preds[:subset_len]

plt.plot(times, currents, label='Current (A)', color='#0284c7', alpha=0.8)
plt.scatter(times[anomalies == 1], currents[anomalies == 1], color='#ef4444', label='Detected Anomaly', s=45, zorder=5)
plt.title('PowerGuard AI — Anomaly Detection on Test Stream')
plt.xlabel('Timestamp')
plt.ylabel('Current (Amperes)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Step 9: Export Model Artifacts for Inference
Serialize the trained model, standard scaler, and deployment metadata containing feature ordering and normalization weights.

In [ ]:
os.makedirs('trained_models', exist_ok=True)
joblib.dump(iso_forest, 'trained_models/isolation_forest_v1.0.joblib')
joblib.dump(scaler, 'trained_models/scaler_v1.0.joblib')

metadata = {
    "model_name": "PowerGuard_IsolationForest_AnomalyDetector",
    "model_version": "v1.0",
    "features": FEATURE_COLUMNS,
    "scaler": {
        "means": [float(m) for m in scaler.mean_],
        "scales": [float(s) for s in scaler.scale_]
    },
    "threshold": float(np.percentile(test_scores, 3.0)),
    "contamination": 0.03
}

with open('trained_models/model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("Artifacts exported to trained_models/ directory successfully!")

## Step 10: Instructions for Reproducing on Kaggle
1. Open [Kaggle Notebooks](https://www.kaggle.com/code).
2. Click **New Notebook** -> Set Environment to **Python 3**, CPU or GPU.
3. Under **Add Data**, search for `uciml/electric-power-consumption` and click **Add**.
4. Upload or copy this notebook and execute all cells sequentially.
5. Download `trained_models/model_metadata.json` and `.joblib` artifacts to incorporate into PowerGuard API.